<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/genaiweek2_d2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#given question:
#chunk dataset into 200 to 500 token segments
#store in Faiss/chroma
#write a function retrieve (query,top_k=3)->return passages

In [ ]:
#dataset->chunking->embeddings->vectore store->retrieve function

In [ ]:
#taking the text data
text_data = """The Amazon rainforest is the largest rainforest in the world, covering an immense area across nine South American countries. It is renowned for its incredible biodiversity, housing millions of species of plants, animals, and insects, many of which are unique to this ecosystem. The Amazon River, flowing through the heart of the rainforest, is the second-longest river globally by length and the largest by discharge volume.

This vast forest plays a crucial role in regulating the Earth's climate by absorbing massive amounts of carbon dioxide. Deforestation, primarily due to agriculture, logging, and mining, poses a significant threat to the Amazon's future and its ability to act as a 'lung of the Earth.' Indigenous communities have lived in the Amazon for thousands of years, relying on its resources and possessing invaluable traditional knowledge about its flora and fauna.

Conservation efforts are ongoing to protect this vital natural resource, involving governments, non-governmental organizations, and local communities. The preservation of the Amazon is not only important for its ecological value but also for its global impact on climate, rainfall patterns, and the livelihoods of its inhabitants."""
print("Sample text dataset defined.")

Sample text dataset defined.


In [ ]:
len(text_data)

1214

In [ ]:
#chunk dataset into 200 to 500 token segments
!pip install langchain_text_splitters

In [ ]:
#chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=80)
docs=text_splitter.create_documents([text_data])
print(f"total number of chunks:{len(docs)}")
print(f"content for first chunk:{docs[0].page_content}")
print(f"length of first chunk:{len(docs[0].page_content)}")

total number of chunks:5
content for first chunk:The Amazon rainforest is the largest rainforest in the world, covering an immense area across nine South American countries. It is renowned for its incredible biodiversity, housing millions of species of plants, animals, and insects, many of which are unique to this ecosystem. The Amazon River, flowing through the heart of the rainforest, is the second-longest river globally by length and the
length of first chunk:395


In [ ]:
#creating embeddings using sentence transformers
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
docs_list=[docs[0],docs[1],docs[2],docs[3],docs[4]]
embeddings=model.encode([doc.page_content for doc in docs_list])
print(f"shape of embeddings:{embeddings.shape}")
print("generated first embedding sample")
#printing the first 5 dimensions of the first embedding
print(embeddings[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


shape of embeddings:(5, 384)
generated first embedding sample
[ 0.14499412  0.00363891 -0.00268584 -0.03898612  0.09555425]


In [ ]:
#store in chroma
!pip install chromadb

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Instantiate the SentenceTransformerEmbeddings model
embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize ChromaDB with the documents and the embeddings model
vectordb = Chroma.from_documents(documents=docs_list, embedding=embeddings_model)

print("ChromaDB initialized with documents and embeddings.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB initialized with documents and embeddings.


In [ ]:
def retrieve(query, top_k=3):
    # Perform a similarity search on ChromaDB
    results = vectordb.similarity_search(query, k=top_k)
    return results

print("Retrieve function defined.")

Retrieve function defined.


In [ ]:
query = "What is the largest rainforest in the world?"
retrieved_docs = retrieve(query, top_k=2)

print(f"\nRetrieved {len(retrieved_docs)} passages for query: '{query}'")
for i, doc in enumerate(retrieved_docs):
    print(f"\nPassage {i+1}:\n{doc.page_content}")


Retrieved 2 passages for query: 'What is the largest rainforest in the world?'

Passage 1:
The Amazon rainforest is the largest rainforest in the world, covering an immense area across nine South American countries. It is renowned for its incredible biodiversity, housing millions of species of plants, animals, and insects, many of which are unique to this ecosystem. The Amazon River, flowing through the heart of the rainforest, is the second-longest river globally by length and the

Passage 2:
The Amazon rainforest is the largest rainforest in the world, covering an immense area across nine South American countries. It is renowned for its incredible biodiversity, housing millions of species of plants, animals, and insects, many of which are unique to this ecosystem. The Amazon River, flowing through the heart of the rainforest, is the second-longest river globally by length and the
